# Download

In [ ]:
!mkdir models

In [ ]:
!git clone -b terrain-test https://github.com/jasper-zheng/streamable-stable-audio-open.git
!mv streamable-stable-audio-open models/

In [ ]:
!git clone https://github.com/jasper-zheng/music2latent-scripted.git
!mv music2latent-scripted models/

In [ ]:
!git clone https://github.com/facebookresearch/FlowDec.git
!mv FlowDec models/

In [ ]:
!wget -O models/VCTK.ts 'https://play.forum.ircam.fr/rave-vst-api/get_model/VCTK'

In [ ]:
!wget -O models/music2latent.pt https://huggingface.co/SonyCSLParis/music2latent/resolve/main/music2latent.pt

In [ ]:
!wget -O models/aam_string_16.ts https://huggingface.co/shuoyang-zheng/jaspers-rave-models/resolve/main/aam_string_b2048_r44100_z16_noncausal.ts

In [ ]:
!wget -O models/aam_drum_16.ts https://huggingface.co/shuoyang-zheng/jaspers-rave-models/resolve/main/aam_drum_b2048_r44100_z16_noncausal.ts



In [ ]:
!wget -O models/FlowDec/checkpoints.zip 'https://github.com/facebookresearch/FlowDec/releases/download/checkpoints/checkpoints.zip'
!unzip models/FlowDec/checkpoints.zip -d models/FlowDec/
!rm models/FlowDec/checkpoints.zip

# Load

In [ ]:
import librosa, torch

device="cpu"
audio_path = librosa.example('trumpet')
wv, sr = librosa.load(audio_path, sr=44100)  
wv = torch.tensor(wv, device=device)
wv = wv.unsqueeze(0).unsqueeze(0)  # add batch dimension


## Music2Latent

In [ ]:
from scripts.factory import Music2LatentWrapper

m2l = Music2LatentWrapper(
    repo_root="models/music2latent-scripted/music2latent",
    package_name="music2latent",
    checkpoint_path="models/music2latent.pt",
    device=device,   # or "mps"/"cuda"
    ratio=4096,
    out_channels=1,
)

In [ ]:
latent = m2l.encode(wv)   # [B, C, T]
wv_rec = m2l.decode(latent)

## Stable Audio Open

In [ ]:
from scripts.factory import StableAudioOpenWrapper

sao = StableAudioOpenWrapper(
    pretrained_id="stabilityai/stable-audio-open-1.0",
    model_half=False,
    skip_bottleneck=True,
    device=device,          # or "mps"/"cuda"
    repo_dir="models/streamable-stable-audio-open",
    use_cached_conv=False, # matches your current setup
)


In [ ]:
latents = sao.encode(wv[:,:,:16384])   # [B, C, T] @ 44.1kHz
wv_rec = sao.decode(latents)

## FlowDec

In [ ]:
from scripts.factory import FlowDecWrapper
# Instantiate once
fd = FlowDecWrapper(
    ckpt_dir="models/FlowDec/checkpoints",
    model_name="flowdec_25s",
    ndac_model="ndac-25",
    n_quantizers=16,
    enhance_steps=3,
    enhance_solver="midpoint",
    device=device,  # or "mps"/"cuda" if available
    flowdec_repo_dir="models/FlowDec",
)

In [ ]:
# Encode from 44.1kHz waveform tensor wv [B, C, T]
zq = fd.encode(wv[:,:,:44100])

# Decode + enhance + return 44.1kHz waveform
rec_44100 = fd.decode(zq)

## RAVE

In [ ]:
rave_vctk = torch.jit.load('models/VCTK.ts').to(device)
rave_vctk = rave_vctk.eval()

with torch.no_grad():
    latents = rave_vctk.encode(wv)
    wv_rec = rave_vctk.decode(latents)

In [ ]:
rave_string = torch.jit.load('models/aam_string_b2048_r44100_z16_noncausal.ts').to(device)
rave_string = rave_string.eval()

with torch.no_grad():
    latents = rave_string.encode(wv)
    wv_rec = rave_string.decode(latents)

# Model Real-Time Factor

In [ ]:
import time
import numpy as np
from fourier_cppn import FourierCPPN

def measure_rtf(encode_fn, terrain_fn, decode_fn, wv, buffer_sizes, comp_ratio, n_warmup: int = 1, n_runs: int = 10):
    device = wv.device
    results = []
    for buf in buffer_sizes:
        T = min(buf, wv.shape[-1])
        assert T//comp_ratio * comp_ratio == T, "Buffer size must be divisible by compression ratio"
        wv_buf = wv[..., :T].to(device)

        # warm-up
        for _ in range(n_warmup):
            _ = encode_fn(wv_buf)

        latents = encode_fn(wv_buf)
        # timed runs
        times = []
        for _ in range(n_runs):
            if terrain_fn is None:
                torch.mps.synchronize()
                start = time.perf_counter()
                _ = decode_fn(latents)
                torch.mps.synchronize()
                end = time.perf_counter()
                
                times.append(end - start)
            else:
                coords = torch.linspace(0.0, 1.0, int(T//comp_ratio)).unsqueeze(1).repeat(1,2).to(device)

                torch.mps.synchronize()
                start = time.perf_counter()
                lt = terrain_fn(coords)
                assert lt.shape == latents.shape, f"Latent channels mismatch, got {lt.shape}, expected {latents.shape}"
                _ = decode_fn(lt)
                torch.mps.synchronize()
                end = time.perf_counter()
                
                times.append(end - start)

        avg_time = float(np.mean(times))
        rtf = avg_time / (T / 44100.0) if avg_time > 0 else float("inf")
        results.append((buf, avg_time, rtf))
    return results

rtf_results = {}


In [ ]:
terrain = FourierCPPN(
            in_dim=2,
            out_dim=64,
            c_max=512,
            gauss_scale=0.05,
            mapping_size=256,
        ).to(device)
def m2l_encode(x):
    return m2l.encode(x)
def m2l_decode(z):
    return m2l.decode(z)
def m2l_terrain(c):
    with torch.no_grad():
        return terrain(c).unsqueeze(0).permute(0,2,1)
comp_ratio = 4096
buffer_sizes = [4096, 8192, 16384, 32768]

results = measure_rtf(m2l_encode, None, m2l_decode, wv, buffer_sizes, comp_ratio, n_warmup=10, n_runs=50)
rtf_results['m2l'] = {}
rtf_results['m2l']['no-terrain'] = results

print("Waiting for a few seconds before next test...")
time.sleep(10)

results = measure_rtf(m2l_encode, m2l_terrain, m2l_decode, wv, buffer_sizes, comp_ratio, n_warmup=10, n_runs=50)
rtf_results['m2l']['with-terrain'] = results

# save results to json
import json
with open(f'outputs/rtf-{device}.json', 'w') as f:
    json.dump(rtf_results, f, indent=4)

In [ ]:
# load results from json
import json
with open(f'outputs/rtf-{device}.json', 'r') as f:
    rtf_results = json.load(f)
rtf_results

In [ ]:
terrain = FourierCPPN(
            in_dim=2,
            out_dim=64,
            c_max=512,
            gauss_scale=0.05,
            mapping_size=256,
        ).to(device)
def sao_encode(x):
    return sao.encode(x)
def sao_decode(z):
    return sao.decode(z)
def sao_terrain(c):
    with torch.no_grad():
        return terrain(c).unsqueeze(0).permute(0,2,1)
comp_ratio = 2048
buffer_sizes = [2048, 4096, 8192, 16384, 32768]

results = measure_rtf(sao_encode, None, sao_decode, wv, buffer_sizes, comp_ratio, n_warmup=10, n_runs=30)
rtf_results['sao'] = {}
rtf_results['sao']['no-terrain'] = results

print("Waiting for a few seconds before next test...")
time.sleep(20)

results = measure_rtf(sao_encode, sao_terrain, sao_decode, wv, buffer_sizes, comp_ratio, n_warmup=10, n_runs=30)
rtf_results['sao']['with-terrain'] = results


# save results to json
import json
with open(f'outputs/rtf-{device}.json', 'w') as f:
    json.dump(rtf_results, f, indent=4)

In [ ]:
# load results from json
import json
with open(f'outputs/rtf-{device}.json', 'r') as f:
    rtf_results = json.load(f)
rtf_results

In [ ]:
terrain = FourierCPPN(
            in_dim=2,
            out_dim=8,
            c_max=512,
            gauss_scale=0.05,
            mapping_size=256,
        ).to(device)
def rave_encode(x):
    with torch.no_grad():
        return rave_vctk.encode(x)
def rave_decode(z):
    with torch.no_grad():
        return rave_vctk.decode(z)
def rave_terrain(c):
    with torch.no_grad():
        return terrain(c).unsqueeze(0).permute(0,2,1)
comp_ratio = 2048
buffer_sizes = [2048, 4096, 8192, 16384, 32768]


results = measure_rtf(rave_encode, None, rave_decode, wv, buffer_sizes, comp_ratio, n_warmup=10, n_runs=50)
rtf_results['rave_vctk'] = {}
rtf_results['rave_vctk']['no-terrain'] = results

print("Waiting for a few seconds before next test...")
time.sleep(20)

results = measure_rtf(rave_encode, rave_terrain, rave_decode, wv, buffer_sizes, comp_ratio, n_warmup=10, n_runs=50)
rtf_results['rave_vctk']['with-terrain'] = results


# save results to json
import json
with open(f'outputs/rtf-{device}.json', 'w') as f:
    json.dump(rtf_results, f, indent=4)

In [ ]:
# load results from json
import json
with open(f'outputs/rtf-{device}.json', 'r') as f:
    rtf_results = json.load(f)
rtf_results

In [ ]:
terrain = FourierCPPN(
            in_dim=2,
            out_dim=16,
            c_max=512,
            gauss_scale=0.05,
            mapping_size=256,
        ).to(device)
def rave_encode(x):
    with torch.no_grad():
        return rave_string.encode(x)
def rave_decode(z):
    with torch.no_grad():
        return rave_string.decode(z)
def rave_terrain(c):
    with torch.no_grad():
        return terrain(c).unsqueeze(0).permute(0,2,1)
comp_ratio = 2048
buffer_sizes = [2048, 4096, 8192, 16384, 32768]


results = measure_rtf(rave_encode, None, rave_decode, wv, buffer_sizes, comp_ratio, n_warmup=10, n_runs=50)
rtf_results['rave_string'] = {}
rtf_results['rave_string']['no-terrain'] = results

print("Waiting for a few seconds before next test...")
time.sleep(10)

results = measure_rtf(rave_encode, rave_terrain, rave_decode, wv, buffer_sizes, comp_ratio, n_warmup=10, n_runs=50)
rtf_results['rave_string']['with-terrain'] = results

# save results to json
import json
with open(f'outputs/rtf-{device}.json', 'w') as f:
    json.dump(rtf_results, f, indent=4)